In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import json

import plotly.graph_objects as go
import numpy as np

In [ ]:
# Font
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['CMU Serif Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 16

In [ ]:
YEAR = 2019
YEAR = 2024

import os
WORKING_DIR = os.environ.get("REPO_ROOT", os.path.abspath(".."))  # repo root (notebooks run from notebooks/)
DATA_DIR = f'{WORKING_DIR}/data'
IMG_DIR = f'{WORKING_DIR}/images'
REGRESSION_DIR = f'{DATA_DIR}/dirichlet/{YEAR}'

In [ ]:
fd = open(f'{DATA_DIR}/parties_description/political_parties_description_europe_2019.json', 'r')
political_parties_description_2019 = json.load(fd)
fd.close()

fd = open(f'{DATA_DIR}/parties_description/political_parties_description_europe_2024.json', 'r')
political_parties_description_2024 = json.load(fd)
fd.close()

In [ ]:
df_r2_2019 = pd.read_pickle(f'{DATA_DIR}/df_r2_scores_2019.pkl')
df_r2_2024 = pd.read_pickle(f'{DATA_DIR}/df_r2_scores_2024.pkl')

In [ ]:
list(df_r2_2019.columns)

In [ ]:
list(df_r2_2024.columns)

In [ ]:
rename_parties_2019 = {'Coal. Renaissance': 'Coal. Besoin d\'Europe',
                       'Coal. Envie d\'Europe': 'Coal. Réveiller l\'Europe'}

df_r2_2019 = df_r2_2019.rename(columns=rename_parties_2019)
common_parties = set(df_r2_2019.columns).intersection(set(df_r2_2024.columns))
common_parties = list(common_parties)

color_party = {}
for party in political_parties_description_2024 :
    party_name = political_parties_description_2024[party]['name']
    color_party[party_name] = political_parties_description_2024[party]['color']

In [ ]:
relationships = {}
    
for party in common_parties:
    relationships[party] = {
        'color': color_party[party],
        2019: {
            'combined': float(df_r2_2019.iloc[0][party]),
            'socioeconomic': float(df_r2_2019.iloc[1][party]),
            'app': float(df_r2_2019.iloc[2][party])
        },
        2024: {
            'combined': float(df_r2_2024.iloc[0][party]),
            'socioeconomic': float(df_r2_2024.iloc[1][party]),
            'app': float(df_r2_2024.iloc[2][party])
        }
    }

In [ ]:
order = ['La France Insoumise',
         'Europe Écologie',
         "Coal. Besoin d'Europe",
         'Les Républicains',
         'Rassemblement National'
         ][::-1]

In [ ]:
change_across_years = {
    'Socioeconomic': {
        2019: np.mean(df_r2_2019.iloc[1]),
        2024: np.mean(df_r2_2024.iloc[1]),
        'relative': (np.mean(df_r2_2024.iloc[1]) - np.mean(df_r2_2019.iloc[1])) / np.mean(df_r2_2019.iloc[1]) * 100
    },
    'Mobile Services': {
        2019: np.mean(df_r2_2019.iloc[2]),
        2024: np.mean(df_r2_2024.iloc[2]),
        'relative': (np.mean(df_r2_2024.iloc[2]) - np.mean(df_r2_2019.iloc[2])) / np.mean(df_r2_2019.iloc[2]) * 100
    },
    'All': {
        2019: np.mean(df_r2_2019.iloc[0]),
        2024: np.mean(df_r2_2024.iloc[0]),
        'relative': (np.mean(df_r2_2024.iloc[0]) - np.mean(df_r2_2019.iloc[0])) / np.mean(df_r2_2019.iloc[0]) * 100
    }
}

In [ ]:
color_2019 = 'tab:blue'
color_2024 = 'steelblue'
text_fontsize = 30
width = 0.35
delta = 0.4


plt.figure(figsize=(18, 14))

ax = plt.gca()

for y in np.arange(0, 0.61, 0.1):
    plt.axhline(y=y, color='gray', linestyle='-', alpha=0.1, zorder=-5)

ax.spines['left'].set_color(color_2019)
ax.yaxis.label.set_color(color_2019)
ax.tick_params(axis='y', colors=color_2019)
plt.ylabel(rf'Adj. $R^2$ Score', color=color_2019, fontsize=text_fontsize, fontweight='bold')

for model_index, model in enumerate(change_across_years):
    model_2019 = change_across_years[model][2019]
    model_2024 = change_across_years[model][2024]

    plt.bar(2*model_index-delta, model_2019, color=color_2019, alpha=0.5, width=width)
    plt.text(2*model_index-delta, .1, '2019', ha='center', va='bottom', rotation=90, fontsize=text_fontsize, color='white', fontweight='bold')
    plt.text(2*model_index-delta, model_2019 + 0.01, f'{model_2019:.2f}', ha='center', fontsize=text_fontsize, alpha=0.75)

    plt.bar(2*model_index, model_2024, color=color_2024, alpha=0.8, width=width)
    plt.text(2*model_index, .1, '2024', ha='center', va='bottom', rotation=90, fontsize=text_fontsize, color='white', fontweight='bold')
    plt.text(2*model_index, model_2024 + 0.01, f'{model_2024:.2f}', ha='center', fontsize=text_fontsize, alpha=0.75)


plt.yticks([-.1, .0, .1, .2, .3, .4, .5, .6], [None, 0, .1, .2, .3, .4, .5, .6], color=color_2019, fontsize=text_fontsize)

plt.axhline(y=0, color='black', linestyle='--', alpha=0.7, zorder=-1)

ax.set_ylim(0, .9)

plt.xticks(ticks=[0, 2, 4], labels=['Socioeconomic', 'Mobile Services', 'All'], fontsize=text_fontsize)
plt.xlim(-.75, 4.75)

ax = plt.twinx()

color = 'tab:green'
ax.spines['right'].set_color(color)
ax.yaxis.label.set_color(color)
ax.tick_params(axis='y', colors=color, width=2)
plt.ylabel(rf'Relative Gain', color=color, fontsize=30, fontweight='bold')


for model_index, model in enumerate(change_across_years):
    relative = change_across_years[model]['relative']

    plt.bar(2*model_index+delta, relative, color='tab:green', alpha=0.8, width=width)

    if relative >= 0:
        plt.text(2*model_index+delta, relative + 0.2, f'{relative:.2f}', ha='center', va='bottom', fontsize=text_fontsize, alpha=0.75)
    else:
        plt.text(2*model_index+delta, relative - 0.5, f'{relative:.2f}', ha='center',  va='top', fontsize=text_fontsize, alpha=0.75)

plt.ylim(-10, 20)
plt.yticks([-10, 0, 10, 20, 30], ['-10%', '0', '10%', '20%', '30%'], color=color, fontsize=text_fontsize)


plt.axhline(y=0, color='black', linestyle='-', alpha=0.7, zorder=-1)

plt.savefig(f'{IMG_DIR}/dirichlet_regression/change_r2_2019_2024.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
parties_change = {}

for party in order:
    r2_socioeconomic_2024 = relationships[party][2024]['socioeconmic']
    r2_socioeconomic_2019 = relationships[party][2019]['socioeconmic']
    percentage_change_socioeconomic = (r2_socioeconomic_2024 - r2_socioeconomic_2019) / r2_socioeconomic_2019 * 100
    # percentage_change_socioeconomic *= 50

    r2_app_2024 = relationships[party][2024]['app']
    r2_app_2019 = relationships[party][2019]['app']
    percentage_change_app = (r2_app_2024 - r2_app_2019) / r2_app_2019 * 100
    # percentage_change_app *= 50 

    parties_change[party] = {
        'socioeconomic': abs(percentage_change_socioeconomic),
        'positive_socioeconomic': percentage_change_socioeconomic > 0,
        'app': abs(percentage_change_app),
        'positive_app': percentage_change_app > 0
    }


In [ ]:
labels = ['Socioeconomic'] + \
            [
                'La France Insoumise',
                'Europe Écologie',
                '''Coal. Réveiller l'Europe<br>(Envie d'Europe)''',
                '''Coal. Besoin d'Europe<br>(Renaissance)''',
                'Les Républicains',
                'Rassemblement National'
            ] + \
         ['Mobile services']

sources = [0]*(len(parties_change)) + [i+1 for i in range(len(parties_change))]
targets = [i+1 for i in range(len(parties_change))] + [len(labels)-1]*(len(parties_change))

values = [parties_change[party]['socioeconomic'] for party in parties_change] + \
         [parties_change[party]['app'] for party in parties_change]

increase_socioeconomic = [parties_change[party]['positive_socioeconomic'] for party in parties_change]
increase_app = [parties_change[party]['positive_app'] for party in parties_change]

positive_color = matplotlib.colors.to_rgb('mediumseagreen')
positive_color = 'rgba' + str(positive_color + (0.5,))
negative_color = matplotlib.colors.to_rgb('salmon')
negative_color = 'rgba' + str(negative_color + (0.5,))


links_color = [negative_color if not increase_socioeconomic[i] else positive_color for i in range(len(parties_change))] + \
              [negative_color if not increase_app[i] else positive_color for i in range(len(parties_change))]
node_color = ['slategrey'] + [relationships[party]['color'] for party in parties_change] + ['slategrey']

In [ ]:
y_ = [-.1, 2.45, 4.9, 7.6, 9, 10.25]

x = [0] + [3.5]*(len(parties_change)) + [10]
y = [5] + y_ + [6]

x = [x[i] / 10 for i in range(len(x))]
y = [y[i] / 10 for i in range(len(y))]

fig = go.Figure(
  layout=go.Layout(
        font_size=30,
        font_color='black',
        width=1100,
        height=700
    ),
  data=[go.Sankey(
    node = dict(
    pad = 15,
    thickness = 20,
    label = labels,
    color = node_color,
    x = x,
    y = y
  ),
  link = dict(
    source = sources, # indices correspond to labels, eg A1, A2, A1, B1, ...
    target = targets,
    value = values,
    color = links_color,
))])

    

fig.show()

In [ ]:
parties_change = {}

labels = [
            'La France Insoumise',
            'Europe Écologie',
            '''Coal. Réveiller l'Europe<br>(Envie d'Europe)''',
            '''Coal. Besoin d'Europe<br>(Renaissance)''',
            'Les Républicains',
            'Rassemblement National'
        ] 



for party in order:
    r2_socioeconomic_2024 = relationships[party][2024]['socioeconmic']
    r2_socioeconomic_2019 = relationships[party][2019]['socioeconmic']
    percentage_change_socioeconomic = (r2_socioeconomic_2024 - r2_socioeconomic_2019) / r2_socioeconomic_2019 * 100
    # percentage_change_socioeconomic *= 50

    r2_app_2024 = relationships[party][2024]['app']
    r2_app_2019 = relationships[party][2019]['app']
    percentage_change_app = (r2_app_2024 - r2_app_2019) / r2_app_2019 * 100
    # percentage_change_app *= 50 

    parties_change[party] = {
        'socioeconomic': abs(percentage_change_socioeconomic),
        'positive_socioeconomic': percentage_change_socioeconomic > 0,
        'app': abs(percentage_change_app),
        'positive_app': percentage_change_app > 0
    }


In [ ]:

labels = [
            'La France Insoumise',
            'Europe Écologie',
            '''Coal. Réveiller l'Europe<br>(Envie d'Europe)''',
            '''Coal. Besoin d'Europe<br>(Renaissance)''',
            'Les Républicains',
            'Rassemblement National'
        ]  +  ['Mobile services'] + ['Socioeconomic']

sources = [i for i in range(len(parties_change))]* 2
targets = [len(labels)-2 for i in range(len(parties_change))] +  [len(labels)-1 for i in range(len(parties_change))]

values = [parties_change[party]['app'] for party in parties_change] + \
        [parties_change[party]['socioeconomic'] for party in parties_change]
         

increase_app = [parties_change[party]['positive_app'] for party in parties_change]
increase_socioeconomic = [parties_change[party]['positive_socioeconomic'] for party in parties_change]

positive_color = matplotlib.colors.to_rgb('mediumseagreen')
positive_color = 'rgba' + str(positive_color + (0.5,))
negative_color = matplotlib.colors.to_rgb('salmon')
negative_color = 'rgba' + str(negative_color + (0.5,))

links_color = [negative_color if not increase_app[i] else positive_color for i in range(len(parties_change))] + \
              [negative_color if not increase_socioeconomic[i] else positive_color for i in range(len(parties_change))]
              
node_color = [relationships[party]['color'] for party in parties_change] + ['slategrey', 'slategrey']

In [ ]:
y_ = [0, 2, 5, 10, 10, 10]

x = [0]*(len(parties_change)) + [8, 10]
y = y_ + [8, 1]

x = [x[i] / 10 for i in range(len(x))]
y = [y[i] / 10 for i in range(len(y))]



In [ ]:
labels[sources[1]], x[1], y[1]

In [ ]:

fig = go.Figure(
  layout=go.Layout(
        font_size=20,
        font_color='black',
        width=1100,
        height=700
    ),
  data=[go.Sankey(
    node = dict(
    pad = 15,
    thickness = 20,
    label = labels,
    color = node_color,
    x = x,
    y = y
  ),
  link = dict(
    source = sources, # indices correspond to labels, eg A1, A2, A1, B1, ...
    target = targets,
    value = values,
    color = links_color,
))])


fig.show()